In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [2]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gradio as gr
from transformers import pipeline

In [3]:
hf_token= userdata.get('HF_TOKEN')
login(hf_token,add_to_git_credential=True)


In [4]:
# pipe1= pipeline("text-generation",
#                 model="openbmb/MiniCPM5-2B",
#                 dtype= torch.float16,
#                 device='cuda',
#                 return_timestamps=True)

# pipe2= pipeline("text-generation",
#                 model="Qwen/Qwen3-0.6B",
#                 dtype= torch.float16,
#                 device='cuda',
#                 return_timestamps=True)


In [5]:
sys_prompt="You are a helpful assistant who generates synthetic data on demand."

# user_p= f"""Genrate data on demand.
# your generated data should include positive, negative and neutral entries on user request:{user_req}
# """

In [6]:
quant_config= BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [7]:
def basic(userp, mdel):
  user_p = f"""Generate data on demand.
  Your generated data should include positive, negative and neutral entries for model training using synthetic data on request: {userp}

  Output ONLY a markdown table with exactly two columns: text | sentiment
  sentiment must be one of: Positive, Negative, Neutral.
  No intro, no explanation, nothing before or after the table.

  Example format:
  | text | sentiment |
  |------|-----------|
  | The service was fantastic! | Positive |
  | It was okay, nothing special. | Neutral |

  Do not use +, -, bullet points, or escape characters anywhere.
  Every row must be exactly: | one complete sentence | Positive/Negative/Neutral |
  """
  messages = [{"role":"system", "content":sys_prompt},
              {"role":"user","content":user_p}]

  model_name = "google/gemma-3-4b-it" if mdel == 1 else "Qwen/Qwen3-0.6B"

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = tokenizer.eos_token
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True).to("cuda")
  streamer = TextStreamer(tokenizer)
  model = AutoModelForCausalLM.from_pretrained(model_name,
                                                device_map="auto") if mdel == 1 else AutoModelForCausalLM.from_pretrained(model_name, #gemma dosent support quantization
                                                device_map="auto",
                                                quantization_config=quant_config)

  output_ids = model.generate(
      inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      streamer=streamer,
      max_new_tokens=3000,
      do_sample=True,
      temperature=0.7,
      repetition_penalty=1.3,
      no_repeat_ngram_size=3
  )

  input_len = inputs["input_ids"].shape[1]

  generated_tokens = output_ids[0][input_len:]
  decoded_generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  return decoded_generated_text

In [8]:
# # result = basic("Generate 10 examples for sentiment analysis on suicide and depression. Each example should be on a new line and formatted as 'Sentiment: [sentiment], Text: [text]'. Include positive, negative, and neutral sentiments (e.g., 4 negative, 3 positive, 3 neutral. Do not halucinate).",1)
# out = basic("customer reviews for a coffee shop, 10 entries", 1)
# # print(out)

The model has generated the data in a text-based format. Now, let's parse it and display it as a pandas DataFrame.

In [9]:
import re
import json
import pandas as pd

def to_table(text: str) -> pd.DataFrame:
    text = text.strip()

    # 1) Markdown table
    md_lines = [l.strip() for l in text.splitlines() if l.strip().startswith("|")]
    if len(md_lines) >= 2:
        parsed = []
        for l in md_lines:
            cells = [c.strip() for c in l.strip("|").split("|")]
            if all(re.fullmatch(r":?-{2,}:?", c) for c in cells if c):  # separator row
                continue
            parsed.append(cells)
        if len(parsed) >= 2:
            header, body = parsed[0], parsed[1:]
            body = [(r + [""] * len(header))[:len(header)] for r in body]  # pad/truncate
            return pd.DataFrame(body, columns=header)

    # 2) JSON list
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group(0))
            if isinstance(data, list) and data and isinstance(data[0], dict):
                return pd.DataFrame(data)
        except json.JSONDecodeError:
            pass

    # 3) Fallback: line-by-line with sentiment labels
    labels = r"(positive|negative|neutral)"
    header_re = re.compile(rf"#*\s*{labels}( entries| examples| reviews| sentences)?\s*:?", re.I)
    trailing_re = re.compile(rf"[\s\(\[\-–|,:]*(?:sentiment|label)?\s*:?\s*{labels}\s*[\)\]]*\.?$", re.I)
    leading_re = re.compile(rf"^(?:sentiment|label)?\s*:?\s*{labels}\s*[:\-–|]\s*", re.I)

    rows, current = [], None
    for line in text.splitlines():
        line = re.sub(r"^\s*(\d+[\.\)]|[-*•])\s+", "", line.strip())  # numbering/bullets
        line = line.replace("**", "").strip()
        if not line:
            continue

        # heading like "Positive:" -> applies to the lines below it
        if header_re.fullmatch(line):
            current = re.search(labels, line, re.I).group(1).capitalize()
            continue

        label = current
        m = trailing_re.search(line)
        if m:
            label = m.group(1).capitalize()
            line = line[:m.start()]
        else:
            m = leading_re.match(line)
            if m:
                label = m.group(1).capitalize()
                line = line[m.end():]

        line = line.strip(" \"'“”")
        if line and label:  # skip intro/outro chatter with no label
            rows.append({"text": line, "sentiment": label})

    if rows:
        return pd.DataFrame(rows)

    # nothing parsed -> don't lose the output
    return pd.DataFrame({"raw_output": [text]})


def basic_table(userp, mdel):
    return to_table(basic(userp, mdel))

In [10]:
def clean_df(df):
    valid = {"positive": "Positive", "negative": "Negative", "neutral": "Neutral"}
    df = df.copy()

    def tidy(s):
        s = str(s).replace("\\n", " ").replace("\\+", "").replace("\\-", "")
        s = re.sub(r"^[\s.+\-–—*_\\]+", "", s)   # leading junk
        s = re.sub(r"[\s\-–—*_\\]+$", "", s)     # trailing junk
        return s.strip()

    df["text"] = df["text"].map(tidy)
    df["sentiment"] = df["sentiment"].map(lambda s: valid.get(str(s).strip().lower()))  # invalid -> NaN

    bad = df[df["sentiment"].isna() | (df["text"] == "")]          # shifted / broken rows
    good = df.dropna(subset=["sentiment"])
    good = good[good["text"] != ""].reset_index(drop=True)
    return good, bad

In [11]:
# # out = basic("customer reviews for a coffee shop, 10 entries", 1)
# print(out)   # should look like a clean markdown table
# df, bad = clean_df(to_table(out))
# display(df); display(bad)

In [12]:
def gradio_generate(prompt, model_choice):
    model_id = 1 if model_choice == "Google Gemma 3-4B-it" else 2
    raw_output = basic(prompt, model_id)

    parsed_df = to_table(raw_output)
    good_df, bad_df = clean_df(parsed_df)

    return raw_output, good_df, bad_df

with gr.Blocks() as demo:
    gr.Markdown("# 🚀 Synthetic Data Generator")
    gr.Markdown("Generate high-quality datasets for model training on demand. Choose your model and prompt below.")

    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Generation Prompt",
                value="customer reviews for a coffee shop, 10 entries",
                placeholder="Type what data you want to generate..."
            )
            model_dropdown = gr.Radio(
                choices=["Google Gemma 3-4B-it", "Qwen 3-0.6B"],
                value="Google Gemma 3-4B-it",
                label="Select Model"
            )
            generate_btn = gr.Button("Generate & Parse", variant="primary")

        with gr.Column():
            with gr.Tab("Parsed Dataset"):
                good_output = gr.Dataframe(label="Valid Structured Data")
                bad_output = gr.Dataframe(label="Malformed / Dropped Entries")
            with gr.Tab("Raw Output"):
                raw_output_box = gr.Textbox(label="Raw Model Text", lines=15)

    generate_btn.click(
        fn=gradio_generate,
        inputs=[prompt_input, model_dropdown],
        outputs=[raw_output_box, good_output, bad_output]
    )

demo.launch(inbrowser=True, share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7fdb04cb7c548955b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<bos><start_of_turn>user
You are a helpful assistant who generates synthetic data on demand.

Generate data on demand.
  Your generated data should include positive, negative and neutral entries for model training using synthetic data on request: customer reviews for a coffee shop, 5 entries

  Output ONLY a markdown table with exactly two columns: text | sentiment
  sentiment must be one of: Positive, Negative, Neutral.
  No intro, no explanation, nothing before or after the table.

  Example format:
  | text | sentiment |
  |------|-----------|
  | The service was fantastic! | Positive |
  | It was okay, nothing special. | Neutral |
  
  Do not use +, -, bullet points, or escape characters anywhere.
  Every row must be exactly: | one complete sentence | Positive/Negative/Neutral |<end_of_turn>
Absolutely. Here's your requested dataset in the specified Markdown table format:

| text                                   | sentiment   |
|----------------------------------------|-----------